# Plate detection - A1 (YOLO26n vs YOLOv8n) - local end-to-end
Runs the full pipeline **locally** (data prep -> train both models x 3 seeds -> imgsz ablation -> ONNX export -> eval + comparison table -> report figures). All logic is in the `plate_detect` package; cells only drive the CLI. Dataset assumed already at `data/raw/kaggle_vn_plate_segment`.

## 0. Install package (editable)

In [ ]:
!pip install -e src/ml/plate_detect

## 0b. Config - edit REPO_ROOT to your local repo path, then run

In [ ]:
REPO_ROOT = "/Users/ducqhle/Documents/workspace/UIT2026-DoAnCuoiKi"  # <- your local repo root
import os
os.chdir(REPO_ROOT)

MODELS = "yolov8n,yolo26n"
SEEDS = "0,1,2"
IMGSZ = 640
ABLATION_IMGSZ = 960
PROJECT = "runs"
WEIGHTS_DIR = "src/ml/plate_detect/weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

import torch
print("device:", "cuda:" + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1. Prepare - class-map gate (BSD/BSV) -> split -> dedup train<->test & train<->val -> validate
> pHash report at `data/processed/a1_det/phash_report.txt` lists near-duplicate frames.

In [ ]:
!plate_detect prepare

In [ ]:
!plate_detect check

## 2. Train - full matrix @640 (both models x seeds 0,1,2)

In [ ]:
!plate_detect train --imgsz {IMGSZ} --seeds {SEEDS} --models {MODELS} --project {PROJECT}

## 3. imgsz ablation @960 (single seed, both models)

In [ ]:
!plate_detect train --imgsz {ABLATION_IMGSZ} --seeds 0 --models {MODELS} --project {PROJECT}

## 4. Export best -> ONNX (per model @640) + copy best.pt to weights/
Adjust the `_s0_` seed per your best val-mAP run if seed 0 is not the best.

In [ ]:
import shutil
for m in ["yolov8n", "yolo26n"]:
    !plate_detect export --weights {PROJECT}/{m}_s0_{IMGSZ}/weights/best.pt --out {WEIGHTS_DIR}/{m}_a1_{IMGSZ}.onnx --imgsz {IMGSZ}
    shutil.copy(f"{PROJECT}/{m}_s0_{IMGSZ}/weights/best.pt", f"{WEIGHTS_DIR}/{m}_a1_{IMGSZ}.pt")
print("exported to", WEIGHTS_DIR)

## 5. Evaluate on A1 test -> comparison table + experiments.csv

In [ ]:
import glob
SAMPLE = sorted(glob.glob("data/processed/a1_det/images/test/*.jpg"))[0]
!plate_detect eval --imgszs {IMGSZ},{ABLATION_IMGSZ} --models {MODELS} --project {PROJECT} --weights-dir {WEIGHTS_DIR} --sample-image {SAMPLE} --csv src/ml/experiments.csv --table docs/report/figures/plate_det_comparison.md

## 6. Report figures - qualitative / low-light / timestamp-FP / class-map visual

In [ ]:
import glob, cv2
from plate_detect.inference.plate_detector import PlateDetector
from plate_detect.data.adapters import A1Adapter
from plate_detect.figures import class_map_grid, qualitative_grid, annotate_and_save

best = f"{WEIGHTS_DIR}/yolo26n_a1_{IMGSZ}.onnx"
det = PlateDetector(best, backend="onnx", conf=0.25)
test = sorted(glob.glob("data/processed/a1_det/images/test/*.jpg"))
qualitative_grid(det, test, "docs/report/figures/plate_det_qualitative.png", n=9)
dark = sorted(test, key=lambda p: cv2.imread(p).mean())[:9]
qualitative_grid(det, dark, "docs/report/figures/plate_det_lowlight.png", n=9)
annotate_and_save(det, test[0], "docs/report/figures/plate_det_timestamp_fp.png")
recs = A1Adapter().read_raw("data/raw/kaggle_vn_plate_segment")
class_map_grid(recs, {0: "bien_1hang", 1: "bien_2hang"}, "docs/report/figures/plate_det_class_map.png", per_class=8)
print("figures written to docs/report/figures/")

## 7. Results

In [ ]:
print(open("docs/report/figures/plate_det_comparison.md").read())
!tail -5 src/ml/experiments.csv